In [ ]:
# !pip install geopy

In [ ]:
import os
import time
import pandas as pd
from bs4 import BeautifulSoup
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager

# SSL 환경 변수 초기화 (PostgreSQL 에러 방지)
for env_var in ['REQUESTS_CA_BUNDLE', 'CURL_CA_BUNDLE']:
    if env_var in os.environ:
        del os.environ[env_var]

def get_twosome_final_boss():
    chrome_options = Options()
    # 진행 상황을 눈으로 확인하기 위해 창을 띄웁니다.
    chrome_options.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36")
    
    driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=chrome_options)
    url = "https://mo.twosome.co.kr/so/storeList.do"
    
    try:
        driver.get(url)
        wait = WebDriverWait(driver, 15)

        # 1. [전체매장] 버튼이 로드될 때까지 기다리고 확실히 클릭
        print("🖱️ '전체매장' 탭으로 전환 중...")
        all_btn_selector = 'a[data="all"]'
        all_store_btn = wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, all_btn_selector)))
        
        # 버튼을 클릭하고 해당 탭이 활성화(보통 class에 on이나 active가 붙음)될 때까지 대기
        driver.execute_script("arguments[0].click();", all_store_btn)
        time.sleep(3) # 탭 전환 후 첫 데이터 로딩 시간 확보

        # 2. 무한 스크롤 (1,700개 목표)
        print("📜 전국 매장 스캔을 시작합니다. 잠시만 기다려 주세요...")
        
        last_count = 0
        retry_limit = 5 # 데이터가 안 늘어날 때 최대 5번까지 재시도 (더 끈질기게)
        no_change_count = 0
        
        while True:
            # 현재 로드된 매장 개수 파악
            current_items = driver.find_elements(By.CSS_SELECTOR, "#infiniteScroll > li[data]")
            current_count = len(current_items)
            
            # 실시간 진행 상황 출력
            print(f"🔄 현재 수집된 매장: {current_count}개...", end="\r")

            if current_count > last_count:
                # 데이터가 늘어났다면 카운트 초기화
                last_count = current_count
                no_change_count = 0
            else:
                # 데이터가 안 늘어난다면
                no_change_count += 1
                
            # 5번 연속으로 숫자가 안 늘어나면 정말 끝인 것으로 판단
            if no_change_count >= retry_limit:
                # 만약 숫자가 너무 적은데(예: 300개 미만) 멈췄다면 강제로 스크롤 한 번 더 시도
                if current_count < 300:
                    driver.execute_script("window.scrollTo(0, 0);") # 위로 갔다가
                    time.sleep(1)
                    driver.execute_script("window.scrollTo(0, document.body.scrollHeight);") # 다시 아래로
                    no_change_count = 0 # 기회 한 번 더
                    continue
                break
            
            # 아래로 끝까지 스크롤
            driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
            time.sleep(2.0) # 서버 응답 대기 시간

        print(f"\n✅ 스캔 종료! 최종 {len(current_items)}개 매장을 찾았습니다.")

        # 3. BeautifulSoup 파싱
        soup = BeautifulSoup(driver.page_source, 'html.parser')
        store_list = soup.select("#infiniteScroll > li[data]")

        results = []
        for store in store_list:
            store_id = store.get('data', '결측치')
            
            # 매장명 추출
            dt = store.find('dt', class_='cf')
            store_nm = dt.find('strong').get_text(strip=True) if dt and dt.find('strong') else "결측치"
            
            # 주소 추출
            dd = store.find('dd', id='addr')
            store_addr = dd.get_text(strip=True) if dd else "결측치"

            results.append({
                "지점ID": store_id,
                "매점명": store_nm,
                "매점주소": store_addr
            })

        return pd.DataFrame(results)

    except Exception as e:
        print(f"\n❌ 오류 발생: {e}")
        return pd.DataFrame()
    finally:
        driver.quit()

# 실행 및 저장
df_twosome = get_twosome_final_boss()

if not df_twosome.empty:
    # 중복 제거 (혹시 모를 중복 수집 방지)
    df_twosome = df_twosome.drop_duplicates(subset=['지점ID'])
    
    print("\n" + "="*50)
    print(f"📊 최종 수집 결과: {len(df_twosome)}개")
    print("="*50)
    print(df_twosome.head(5))
    
    df_twosome.to_csv("twosome_place.csv", index=False, encoding="utf-8-sig")
    print("\n💾 'twosome_place.csv' 파일로 저장되었습니다.")

🖱️ '전체매장' 탭으로 전환 중...
📜 전국 매장 스캔을 시작합니다. 잠시만 기다려 주세요...
🔄 현재 수집된 매장: 1731개...
✅ 스캔 종료! 최종 1731개 매장을 찾았습니다.

📊 최종 수집 결과: 1731개
      지점ID          매점명                                               매점주소
0  1903894      전농동사거리점                        서울특별시 동대문구 사가정로 97 (전농동) 1층
1  1903871  파주오두산통일전망대점                  경기도 파주시 탄현면 필승로 369 (오두산통일전망대) 4층
2  1903869    인천영종한빛타워점        인천광역시 중구 쪽빛하늘로 72 (운남동) 한빛타워 106, 107, 108호
3  1903788         홍제역점  서울특별시 서대문구 통일로 455 (홍제동, 경인주유소) 서울시 서대문구  홍제동 ...
4  1903851      대구지산광장점               대구광역시 수성구 용학로 297-7 (지산동) 1,2층 (지산동)

💾 'twosome_final_data.csv' 파일로 저장되었습니다.


In [1]:
import os
import time
import random
import pandas as pd
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager

# 1. SSL 인증서 에러 방지 (지긋지긋한 PostgreSQL 경로 에러 해결)
for env_var in ['REQUESTS_CA_BUNDLE', 'CURL_CA_BUNDLE']:
    if env_var in os.environ:
        del os.environ[env_var]

def get_phone_with_selenium():
    file_path = "twosome_place.csv"
    if not os.path.exists(file_path):
        print("❌ 'twosome_place.csv' 파일이 없습니다.")
        return

    df = pd.read_csv(file_path)
    
    # '연결실패'이거나 아직 번호가 없는 행만 골라내기
    target_indices = df[df['전화번호'].isin(['연결실패', '연결실패 ', None]) | df['전화번호'].isna()].index
    
    if len(target_indices) == 0:
        print("✅ 모든 전화번호가 수집되었습니다!")
        return

    print(f"🕵️ 총 {len(target_indices)}개의 '연결실패' 매장을 Selenium으로 재시도합니다.")

    # 브라우저 설정 (최대한 사람처럼 보이게)
    chrome_options = Options()
    # 차단을 피하기 위해 창을 띄우는 것을 권장하지만, 너무 번거로우면 headless를 켜세요.
    # chrome_options.add_argument("--headless") 
    chrome_options.add_argument("--disable-blink-features=AutomationControlled") # 자동화 감지 우회
    chrome_options.add_experimental_option("excludeSwitches", ["enable-automation"])
    chrome_options.add_experimental_option('useAutomationExtension', False)
    chrome_options.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/123.0.0.0 Safari/537.36")

    driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=chrome_options)
    
    # 봇 감지 우회를 위해 웹드라이버 속성 제거
    driver.execute_cdp_cmd("Page.addScriptToEvaluateOnNewDocument", {
        "source": "Object.defineProperty(navigator, 'webdriver', {get: () => undefined})"
    })

    base_url = "https://mo.twosome.co.kr/so/storeDetail.do?storeCd={}"

    try:
        # 진행률을 위해 enumerate 사용
        for i, idx in enumerate(target_indices):
            store_id = df.at[idx, '지점ID']
            url = base_url.format(store_id)
            
            try:
                driver.get(url)
                # 페이지 로딩 대기 (td 태그가 나타날 때까지 최대 10초)
                wait = WebDriverWait(driver, 10)
                
                # 전화번호 링크(tel:)가 포함된 a 태그 찾기
                phone_el = wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, 'a[href^="tel:"]')))
                phone_number = phone_el.text.strip()
                
                df.at[idx, '전화번호'] = phone_number
                print(f"[{i+1}/{len(target_indices)}] {df.at[idx, '매점명']}: {phone_number}")

            except Exception:
                print(f"[{i+1}/{len(target_indices)}] {df.at[idx, '매점명']}: 여전히 실패")
                df.at[idx, '전화번호'] = "연결실패"

            # 🛑 중요: 사람처럼 보이게 랜덤하게 쉽니다.
            time.sleep(random.uniform(1.5, 3.5))

            # 30개 수집할 때마다 브라우저를 껐다 켜거나 1분간 휴식 (차단 방지 핵심)
            if (i + 1) % 30 == 0:
                print("\n💤 서버 의심을 피하기 위해 1분간 휴식합니다...")
                time.sleep(60)
                # 100개마다 중간 저장
                df.to_csv(file_path, index=False, encoding="utf-8-sig")

    finally:
        df.to_csv(file_path, index=False, encoding="utf-8-sig")
        driver.quit()
        print(f"\n✅ 작업 완료! '{file_path}'가 업데이트되었습니다.")

# 실행
get_phone_with_selenium()

🕵️ 총 816개의 '연결실패' 매장을 Selenium으로 재시도합니다.
[1/816] 대전대서문점: 042-283-5633
[2/816] 구미옥계점: 054-475-0860
[3/816] 신내SKV1센터점: 02-6252-5321
[4/816] 서일대점: 02-2208-1030
[5/816] 부천중동위브점: 032-620-5812
[6/816] 군산지곡호수공원점: 063-463-8700
[7/816] 고양삼송미디어시티점: 02-2039-9122
[8/816] 경희대국제캠퍼스점: 031-203-2316
[9/816] 도봉산입구점: 02-3492-7778
[10/816] 용인처인구청점: 031-334-0770
[11/816] 전주반월DT점: 063-212-6600
[12/816] 대전원신흥점: 042-825-5778
[13/816] 대전목원대점: 042-826-1607
[14/816] 송파잠실H타워점: 02-417-4360
[15/816] 서판교점: 031-705-0778
[16/816] 별내용암천점: 031-571-1222
[17/816] 제주오라우정점: 064-747-3637
[18/816] 대구반월당점: 053-253-3356
[19/816] 양평양수점: 031-775-0722
[20/816] 청계천광교점: 02-2135-5622
[21/816] 진주세란병원점: 187-7-0112
[22/816] 광주두암타운점: 062-266-3388
[23/816] 제일제당사옥점: 02-2039-2188
[24/816] 기흥AK몰점: 031-693-6087
[25/816] 제천고암케이원점: 043-645-2222
[26/816] 경산대로점: 053-792-9295
[27/816] 대구월배아이파크점: 053-285-3424
[28/816] 서판교운중타운점: 031-8039-7970
[29/816] 목포석현점: 061-801-7589
[30/816] 대전판암역점: 042-273-0888

💤 서버 의심을 피하기 위해 1분간 휴식합니다...
[31/816] 일산덕이점: 031

In [2]:
import pandas as pd
import os

def cleanup_twosome_csv():
    file_path = "twosome_place.csv"
    
    if not os.path.exists(file_path):
        print(f"❌ '{file_path}' 파일이 없습니다.")
        return

    # 1. CSV 로드
    df = pd.read_csv(file_path)
    original_count = len(df)

    # 2. 결측치 및 '실패' 데이터 제거
    # NaN(비어있는 값) 제거
    df = df.dropna(subset=['경도', '위도'])
    
    # '실패'라고 적힌 문자열 제거 (공백 포함 여부 확인)
    df = df[df['경도'].astype(str).str.strip() != "실패"]
    df = df[df['위도'].astype(str).str.strip() != "실패"]

    cleaned_count = len(df)
    removed_count = original_count - cleaned_count

    # 3. 결과 저장 (기존 파일 덮어쓰기)
    df.to_csv(file_path, index=False, encoding="utf-8-sig")
    
    print(f"✨ 정리 완료!")
    print(f"📊 기존 데이터: {original_count}개")
    print(f"🗑️ 삭제된 데이터: {removed_count}개")
    print(f"✅ 최종 데이터: {cleaned_count}개")

if __name__ == "__main__":
    cleanup_twosome_csv()

✨ 정리 완료!
📊 기존 데이터: 1731개
🗑️ 삭제된 데이터: 1개
✅ 최종 데이터: 1730개


In [7]:
import pandas as pd
from geopy.geocoders import Nominatim
from geopy.extra.rate_limiter import RateLimiter
import time
import os
import re
from datetime import datetime
from sqlalchemy import create_engine, text

# --- [설정 정보] ---
DB_USER, DB_PASS = "root", "1234"
DB_HOST, DB_PORT, DB_NAME = "localhost", "3306", "coffee_store"
CSV_FILE = "twosome_place.csv"

def clean_address(addr):
    """주소에서 층수, 호수, 괄호 등 불필요한 정보 제거"""
    if not addr or pd.isna(addr): return ""
    # 1. 괄호와 그 안의 내용 제거: (전농동) -> 삭제
    addr = re.sub(r'\(.*?\)', '', addr)
    # 2. 상세 층수 및 호수 제거 (숫자+층, 숫자+호 등)
    addr = re.sub(r'\d+층|\d+호|지하\d+층|지상\d+층', '', addr)
    # 3. 불필요한 공백 정리
    return " ".join(addr.split())

def run_smart_geocoder():
    if not os.path.exists(CSV_FILE):
        print(f"❌ '{CSV_FILE}' 파일이 없습니다.")
        return

    df = pd.read_csv(CSV_FILE, encoding='utf-8-sig')
    df.columns = df.columns.str.strip()

    # 결과 저장용 컬럼 생성 (기존 데이터 유지)
    if '위도' not in df.columns: df['위도'] = None
    if '경도' not in df.columns: df['경도'] = None

    geolocator = Nominatim(user_agent="twosome_final_bot")
    geocode = RateLimiter(geolocator.geocode, min_delay_seconds=1, max_retries=2)

    print(f"[{datetime.now().strftime('%H:%M:%S')}] 🚀 정밀 지오코딩을 시작합니다.")
    print("-" * 60)

    for i, row in df.iterrows():
        # 이미 좌표가 있는 행은 건너뜁니다 (이어서 하기 가능)
        if pd.notna(row['위도']) and pd.notna(row['경도']):
            continue

        store_nm = row['매점명']
        original_addr = row['매점주소']
        
        # 주소 정제 실행
        refined_addr = clean_address(original_addr)
        
        try:
            # 1차 시도: 정제된 주소로 검색
            location = geocode(refined_addr)
            
            # 2차 시도: 실패 시 더 짧게 (앞에서 3~4단어만) 잘라서 시도
            if not location:
                short_addr = " ".join(refined_addr.split()[:4])
                location = geocode(short_addr)

            if location:
                df.at[i, '위도'] = location.latitude
                df.at[i, '경도'] = location.longitude
                status = "✅ 성공"
            else:
                status = "❌ 실패"
        except Exception as e:
            status = f"⚠️ 에러"

        now = datetime.now().strftime('%H:%M:%S')
        print(f"[{now}] ({i+1}/{len(df)}) {store_nm[:8]:<8} | {status} | 검색주소: {refined_addr[:25]}...")

        # 10건마다 CSV 중간 저장 (안정성)
        if (i + 1) % 10 == 0:
            df.to_csv(CSV_FILE, index=False, encoding="utf-8-sig")

    # 최종 결과 저장 및 DB 전송
    df.to_csv(CSV_FILE, index=False, encoding="utf-8-sig")
    print("-" * 60)
    print(f"✅ 모든 변환 종료. DB 전송을 시작합니다.")

    try:
        engine = create_engine(f"mysql+pymysql://{DB_USER}:{DB_PASS}@{DB_HOST}:{DB_PORT}/{DB_NAME}?charset=utf8mb4")
        with engine.begin() as con:
            con.execute(text("TRUNCATE TABLE twosome"))
            df.dropna(subset=['위도', '경도']).to_sql(name='twosome', con=con, if_exists='append', index=False)
        print(f"🎉 MySQL 반영 완료!")
    except Exception as e:
        print(f"❌ DB 오류: {e}")

if __name__ == "__main__":
    run_smart_geocoder()

[14:40:26] 🚀 정밀 지오코딩을 시작합니다.
------------------------------------------------------------
[14:40:26] (1/1730) 전농동사거리점  | ✅ 성공 | 검색주소: 서울특별시 동대문구 사가정로 97...
[14:40:27] (2/1730) 파주오두산통일전 | ✅ 성공 | 검색주소: 경기도 파주시 탄현면 필승로 369...
[14:40:29] (3/1730) 인천영종한빛타워 | ❌ 실패 | 검색주소: 인천광역시 중구 쪽빛하늘로 72 한빛타워 10...
[14:40:31] (4/1730) 홍제역점     | ✅ 성공 | 검색주소: 서울특별시 서대문구 통일로 455 서울시 서대...
[14:40:33] (5/1730) 대구지산광장점  | ✅ 성공 | 검색주소: 대구광역시 수성구 용학로 297-7 1,...
[14:40:35] (6/1730) 아산CGV점   | ✅ 성공 | 검색주소: 충청남도 아산시 온천대로 1538 투썸플레이스...
[14:40:36] (7/1730) 강남구청역점   | ✅ 성공 | 검색주소: 서울 강남구 학동로 335...
[14:40:37] (8/1730) 남해창선점    | ✅ 성공 | 검색주소: 경남 남해군 창선면 대벽리 15...
[14:40:38] (9/1730) 인천공항세인물류 | ✅ 성공 | 검색주소: 인천광역시 중구 자유무역로 25...
[14:40:39] (10/1730) 세종신흥점    | ✅ 성공 | 검색주소: 세종특별자치시 조치원읍 신흥리 166-11...
[14:40:40] (11/1730) 인천영종운서점  | ✅ 성공 | 검색주소: 인천광역시 중구 햇내로28번길 10...
[14:40:43] (12/1730) 내포신도시점   | ✅ 성공 | 검색주소: 충청남도 홍성군 홍북읍 홍학로 91 투썸플레이...
[14:40:44] (13/1730) 판교봇들마을점  | ✅ 성공 | 검색주소: 경기 성남시 분당구 삼평동 721 스타식스코아...
[14:40:4

In [ ]:
import pandas as pd
import requests
import time
import os
import re
from datetime import datetime
from sqlalchemy import create_engine, text
import urllib3

# [1] SSL 인증서 경로 에러 방지 및 경고 무시
for env_var in ['REQUESTS_CA_BUNDLE', 'CURL_CA_BUNDLE']:
    if env_var in os.environ:
        del os.environ[env_var]
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

# --- [설정 정보] ---
KAKAO_API_KEY = "Secret"
DB_USER, DB_PASS = "root", "본인비밀번호"
DB_HOST, DB_PORT, DB_NAME = "localhost", "3306", "coffee_store"
CSV_FILE = "twosome_place.csv"

def clean_address(addr):
    """주소에서 지도 API가 인식하기 힘든 상세 정보 제거"""
    if not addr or pd.isna(addr): return ""
    # 괄호 및 내용 제거 (예: (전농동))
    addr = re.sub(r'\(.*?\)', '', addr)
    # 상세 층수 및 호수 제거 (예: 1층, 지하2층, 101호)
    addr = re.sub(r'\d+층|\d+호|지하\d+층|지상\d+층', '', addr)
    return " ".join(addr.split())

def get_kakao_coords(address):
    """카카오 API 호출"""
    url = "https://dapi.kakao.com/v2/local/search/address.json"
    headers = {"Authorization": f"KakaoAK {KAKAO_API_KEY}"}
    params = {"query": address}
    try:
        res = requests.get(url, headers=headers, params=params, timeout=10, verify=False)
        if res.status_code == 200:
            data = res.json()
            if data['documents']:
                return float(data['documents'][0]['y']), float(data['documents'][0]['x'])
        return None, None
    except:
        return None, None

def run_failed_data_recovery():
    if not os.path.exists(CSV_FILE):
        print(f"❌ '{CSV_FILE}' 파일이 없습니다.")
        return

    # 1. 데이터 로드
    df = pd.read_csv(CSV_FILE, encoding='utf-8-sig')
    df.columns = df.columns.str.strip()

    # '위도' 컬럼이 없으면 생성, 있으면 빈 값 확인
    if '위도' not in df.columns:
        df['위도'] = None
        df['경도'] = None

    # 2. 실패한 데이터만 필터링 (NaN 혹은 'None' 문자열 등)
    failed_mask = df['위도'].isna() | (df['위도'] == '') | (df['위도'].astype(str) == 'None')
    failed_indices = df[failed_mask].index

    if len(failed_indices) == 0:
        print("✅ 모든 데이터에 이미 좌표가 존재합니다.")
        return

    print(f"[{datetime.now().strftime('%H:%M:%S')}] 🛠️ 실패 데이터 {len(failed_indices)}건에 대해 카카오 API 보정을 시작합니다.")
    print("-" * 70)

    # 3. 보정 작업 수행
    for i in failed_indices:
        store_nm = df.at[i, '매점명']
        original_addr = df.at[i, '매점주소']
        
        # 주소 정제 후 API 호출
        refined_addr = clean_address(original_addr)
        lat, lon = get_kakao_coords(refined_addr)
        
        # 1차 실패 시 주소를 더 짧게 잘라서 재시도 (건물명 제외)
        if lat is None:
            short_addr = " ".join(refined_addr.split()[:4])
            lat, lon = get_kakao_coords(short_addr)

        if lat:
            df.at[i, '위도'] = lat
            df.at[i, '경도'] = lon
            status = "✅ 보정성공"
        else:
            status = "❌ 최종실패"

        now = datetime.now().strftime('%H:%M:%S')
        print(f"[{now}] ({i+1}/{len(df)}) {store_nm[:10]:<10} | {status} | 주소: {refined_addr[:25]}...")
        
        time.sleep(0.05)

    # 4. 결과 저장
    df.to_csv(CSV_FILE, index=False, encoding="utf-8-sig")
    print("-" * 70)
    print(f"[{datetime.now().strftime('%H:%M:%S')}] ✅ 보정 작업 완료 및 CSV 저장.")

    # 5. MySQL 최종 업데이트
    try:
        engine = create_engine(f"mysql+pymysql://{DB_USER}:{DB_PASS}@{DB_HOST}:{DB_PORT}/{DB_NAME}?charset=utf8mb4")
        with engine.begin() as con:
            con.execute(text("TRUNCATE TABLE twosome"))
            df.dropna(subset=['위도', '경도']).to_sql(name='twosome', con=con, if_exists='append', index=False)
        print(f"🎉 MySQL 반영 완료!")
    except Exception as e:
        print(f"❌ DB 오류: {e}")

if __name__ == "__main__":
    run_failed_data_recovery()

[15:21:18] 🛠️ 실패 데이터 255건에 대해 카카오 API 보정을 시작합니다.
----------------------------------------------------------------------
[15:21:19] (3/1730) 인천영종한빛타워점  | ✅ 보정성공 | 주소: 인천광역시 중구 쪽빛하늘로 72 한빛타워 10...
[15:21:19] (25/1730) 안산각골공원점    | ✅ 보정성공 | 주소: 경기도 안산시 상록구 각골로 63...
[15:21:20] (30/1730) 부천옥길점      | ✅ 보정성공 | 주소: 경기도 부천시 옥길로 110-11, 104,1...
[15:21:20] (32/1730) 울산송정점      | ✅ 보정성공 | 주소: 울산광역시 북구 박상진11로 1...
[15:21:20] (35/1730) 마산합성산호천점   | ✅ 보정성공 | 주소: 경상남도 창원시 마산회원구 양덕옛3길 37...
[15:21:21] (42/1730) 함안휴게소(순천)점 | ✅ 보정성공 | 주소: 경상남도 함안군 군북면 현포로 205-1 함안...
[15:21:21] (43/1730) 음성성본DI점    | ✅ 보정성공 | 주소: 충북 음성군 대소면 성본리 907 투썸플레이스...
[15:21:21] (44/1730) 파인애비뉴점     | ✅ 보정성공 | 주소: 서울 중구 을지로100...
[15:21:21] (46/1730) 분당수내점      | ✅ 보정성공 | 주소: 경기도 성남시 분당구 황새울로258번길 31 ...
[15:21:22] (66/1730) 공주동학사점     | ✅ 보정성공 | 주소: 충남 공주시 반포면 학봉리 716-1...
[15:21:22] (67/1730) 고창고인돌휴게소(서 | ✅ 보정성공 | 주소: 전북특별자치도 고창군 고창읍 서해안고속도로 8...
[15:21:22] (68/1730) 고창고인돌휴게소(목 | ✅ 보정성공 | 주소: 전북특별자치도 고창군 신림면 서해안고속도로 8...
[15:

In [9]:
import pandas as pd
from sqlalchemy import create_engine, text
import os

# --- [설정 정보] 본인의 정보로 수정해 주세요 ---
DB_USER, DB_PASS = "root", "1234"
DB_HOST, DB_PORT, DB_NAME = "localhost", "3306", "coffee_store"
CSV_FILE = "twosome_place.csv"

def final_db_sync():
    if not os.path.exists(CSV_FILE):
        print(f"❌ '{CSV_FILE}' 파일이 없습니다.")
        return

    # 1. CSV 데이터 로드
    df = pd.read_csv(CSV_FILE, encoding='utf-8-sig')
    
    # 2. 위도/경도 값이 있는 데이터만 추출 (실패한 데이터 제외)
    # df['위도']가 숫자형인지 확인하고 결측치 제거
    df['위도'] = pd.to_numeric(df['위도'], errors='coerce')
    df['경도'] = pd.to_numeric(df['경도'], errors='coerce')
    final_df = df.dropna(subset=['위도', '경도'])

    print(f"📊 전송 준비 완료: 총 {len(final_df)}개의 매장 좌표 데이터를 전송합니다.")

    # 3. MySQL 연결 및 전송
    try:
        # DB 연결 엔진 생성
        engine = create_engine(f"mysql+pymysql://{DB_USER}:{DB_PASS}@{DB_HOST}:{DB_PORT}/{DB_NAME}?charset=utf8mb4")
        
        with engine.begin() as con:
            # [핵심] 기존 테이블이 있다면 삭제하고, 현재 데이터프레임 구조에 맞춰 새로 생성 (if_exists='replace')
            # 이렇게 하면 '위도', '경도' 컬럼이 자동으로 생성됩니다.
            final_df.to_sql(name='twosome', con=con, if_exists='replace', index=False)
            
            # 테이블 구조 최적화 (지점ID를 기본키로 설정)
            con.execute(text("ALTER TABLE twosome MODIFY COLUMN 지점ID INT;"))
            con.execute(text("ALTER TABLE twosome ADD PRIMARY KEY (지점ID);"))
            
            # 위도, 경도 타입을 DOUBLE로 변경 (정밀도 유지)
            con.execute(text("ALTER TABLE twosome MODIFY COLUMN 위도 DOUBLE;"))
            con.execute(text("ALTER TABLE twosome MODIFY COLUMN 경도 DOUBLE;"))

        print("🎉 [성공] MySQL 테이블이 새로 생성되었고 데이터가 모두 들어갔습니다!")
        print(f"나머지 수집 실패한 데이터는 '{CSV_FILE}'에 위도/경도가 비어있는 채로 남아있습니다.")

    except Exception as e:
        print(f"❌ DB 작업 중 오류 발생: {e}")

if __name__ == "__main__":
    final_db_sync()

📊 전송 준비 완료: 총 1720개의 매장 좌표 데이터를 전송합니다.
🎉 [성공] MySQL 테이블이 새로 생성되었고 데이터가 모두 들어갔습니다!
나머지 수집 실패한 데이터는 'twosome_place.csv'에 위도/경도가 비어있는 채로 남아있습니다.


In [10]:
import pandas as pd

# 1. 파일 불러오기
CSV_FILE = "twosome_place.csv"
df = pd.read_csv(CSV_FILE, encoding='utf-8-sig')

# 컬럼명 공백 제거 (혹시 모를 에러 방지)
df.columns = df.columns.str.strip()

# 2. 위도와 경도를 숫자형으로 변환 (비어있거나 문자열인 경우 NaN으로 처리)
df['위도'] = pd.to_numeric(df['위도'], errors='coerce')
df['경도'] = pd.to_numeric(df['경도'], errors='coerce')

# 3. 위도 또는 경도가 NaN(결측치)인 행만 골라내기
missing_df = df[df['위도'].isna() | df['경도'].isna()]

# 4. 결과 출력
print("--- [좌표 결측치 확인 결과] ---")
print(f"전체 데이터: {len(df)}건")
print(f"결측 데이터: {len(missing_df)}건")
print("-" * 30)

if len(missing_df) > 0:
    # 결측치가 있는 데이터의 매점명과 주소만 보여줍니다.
    # 데이터가 너무 많으면 상위 일부만 출력됩니다.
    print(missing_df[['매점명', '매점주소']])
else:
    print("✅ 모든 데이터에 위도/경도 값이 채워져 있습니다.")

--- [좌표 결측치 확인 결과] ---
전체 데이터: 1730건
결측 데이터: 10건
------------------------------
                매점명                                   매점주소
513           부산신호점                   부산 강서구신호산단3로, 48  1층
983           부천괴안점    경기도 부천시 경인로536번길 23 (괴안동) 104호~107호
1193  증평블랙스톤벨포레리조트점                     충북 증평군 벼루재길 512-15
1259          동탄산척점       경기 화성시 동탄2지구 근상45블럭 2롯트 8312-202
1290        창원시티세븐점  경남 창원시 의창구 원이대로320, 더시티세븐 W동 지상층 206호
1338          진해용원점     경남 창원시 진해구 4청안로 241-61, 외 3필지(안골동)
1420          대전법원점                   대전 서구 둔산로137번지 36 1층
1586  청주지웰시티CGV스윗바점                    충북 청주시 홍덕구 대농로 17-0
1597          일산화정점            경기도 고양시 덕양구 화중로72,1,2층(화정동)
1681          대구앞산점              대구광역시 남구 현충로 24,1,2층(대명동)


In [ ]:
## 수동으로 지점 위도, 경도 채우기
import pandas as pd

# 1. 파일 불러오기
df = pd.read_csv("twosome_place.csv", encoding='utf-8-sig')

# 2. 수동으로 채울 매장 리스트 (예시입니다. 실제 10개 지점을 적어주세요)
manual_data = {
    "매점명": ["지점A", "지점B", "지점C"], # 찾으신 매점명 정확히 입력
    "위도": [37.1234567, 37.2345678, 37.3456789],
    "경도": [127.1234567, 127.2345678, 127.3456789]
}

# 3. 데이터프레임 업데이트
for name, lat, lon in zip(manual_data["매점명"], manual_data["위도"], manual_data["경도"]):
    df.loc[df['매점명'] == name, ['위도', '경도']] = [lat, lon]

# 4. 저장
df.to_csv("twosome_place.csv", index=False, encoding='utf-8-sig')
print("✅ 수동 보정이 완료되었습니다!")